# DATA 612 — Final Project Planning Document
**Zoran Glisovic**
**DATA 612 — Recommender Systems, Summer 2026**

---

Projects 1 through 5 built up a single recommender system on the Netflix Prize dataset, one method at a time: global baseline predictors, content-based and collaborative filtering, matrix factorization via SVD, precision/recall/coverage beyond RMSE, and finally ALS on Spark. Each project used a progressively larger slice of the same data, but every slice so far has been a curated subset -- the most-rated movies, restricted to users with enough ratings to make the sparsity manageable.

The Final Project uses the complete dataset instead: all four `combined_data` files, every user, every movie, no filtering. That scale is itself the "additional data" this assignment calls for, since every prior project in this series worked from a deliberately narrowed slice of the same corpus. On top of the full data, this project adds a temporal component -- ratings are not stationary over time, and modeling that directly is the planned addition beyond straightforward matrix factorization.

In [1]:
import os
import time
from collections import defaultdict

import numpy as np
import pandas as pd

SEED = 45   # same fixed seed used across this project's notebooks for reproducibility
np.random.seed(SEED)
DATA_PATH = '../Resources/Netflix Prize data/'

combined_files = ['combined_data_1.txt', 'combined_data_2.txt', 'combined_data_3.txt', 'combined_data_4.txt']

total_ratings = 0
unique_users, unique_movies = set(), set()
min_date, max_date = None, None
per_file_counts = {}

for fname in combined_files:
    file_ratings = 0
    with open(os.path.join(DATA_PATH, fname), 'r') as f:
        for line in f:
            line = line.strip()
            if line.endswith(':'):
                unique_movies.add(int(line[:-1]))
            else:
                user_id, rating, date = line.split(',')
                unique_users.add(int(user_id))
                file_ratings += 1
                if min_date is None or date < min_date:
                    min_date = date
                if max_date is None or date > max_date:
                    max_date = date
    per_file_counts[fname] = file_ratings
    total_ratings += file_ratings

for fname, count in per_file_counts.items():
    print(f'{fname:<22} {count:>12,} ratings')
print()
print(f'Total ratings:     {total_ratings:>12,}')
print(f'Unique users:      {len(unique_users):>12,}')
print(f'Unique movies:     {len(unique_movies):>12,}')
print(f'Date range:        {min_date} to {max_date}')
sparsity = 1 - total_ratings / (len(unique_users) * len(unique_movies))
print(f'Sparsity:          {sparsity:.4%}')
print(f'Avg ratings/user:  {total_ratings/len(unique_users):.2f}')
print(f'Avg ratings/movie: {total_ratings/len(unique_movies):,.2f}')

combined_data_1.txt      24,053,764 ratings
combined_data_2.txt      26,977,591 ratings
combined_data_3.txt      22,601,629 ratings
combined_data_4.txt      26,847,523 ratings

Total ratings:      100,480,507
Unique users:           480,189
Unique movies:           17,770
Date range:        1999-11-11 to 2005-12-31
Sparsity:          98.8224%
Avg ratings/user:  209.25
Avg ratings/movie: 5,654.50


## 1. The Dataset: Full Netflix Prize Corpus

The four `combined_data` files together hold every rating Netflix released for the Prize -- no filtering by movie popularity or user activity, unlike every previous project in this series. Each file uses the same block format as before: a line with just a movie id and a colon, followed by one `CustomerID,Rating,Date` line per rating for that movie, repeated for the next movie, and so on. `movie_titles.csv` maps movie id to release year and title. `probe.txt` and `qualifying.txt` are the two evaluation files that shipped with the original competition data, and they're the reason this project can replicate the actual Prize methodology rather than inventing a new train/test split from scratch.

A full pass over all four files (not a sample) confirms the real scope of this dataset: **100,480,507 ratings** from **480,189 unique users** across **17,770 unique movies**, spread across roughly six years of activity -- matching the training set size documented for this dataset. That match is a useful sanity check before building anything on top of it: the local files are complete and unmodified, and the parser logic already proven out in Project 5 (read the full file, track the current movie id across a colon-terminated header line, append one row per rating) carries over directly, just pointed at four files instead of one and with the popularity/activity filters removed entirely.

Two things stand out against every prior subset in this series. First, the sparsity is noticeably higher -- 98.82%, versus Project 5's 96.61% on a popularity-filtered subset -- which is the direct consequence of including every user and movie rather than only the well-represented ones; average ratings per user (209) and per movie (5,655) look generous until it's clear how unevenly that activity is actually distributed across 480,189 users and 17,770 movies. Second, the actual earliest rating date found by this scan is **1999-11-11**, not October 1998 as the dataset's own README describes -- a discrepancy worth flagging honestly rather than quietly using whichever number was convenient, since it's a real mismatch between the documentation and the data actually shipped.

## 2. Evaluation Protocol: Replicating the Actual Prize Methodology

Rather than building a new random train/test split the way Projects 1-5 did, this project uses the same two evaluation files the actual competition shipped with:

- **`probe.txt`** lists (movie id, customer id) pairs whose true ratings already exist somewhere inside the `combined_data` files. The standard approach -- and the one used here -- is to hold those exact pairs out of training, fit the model on everything else, and then score RMSE by comparing predictions against the true ratings that were held back. This gives a genuine, reusable held-out test set without inventing a new split.
- **`qualifying.txt`** is the file the actual competition scored submissions against, and Netflix never published the true ratings for it after the competition closed. A prediction file can still be generated in the exact required format (movie id, then one predicted rating per customer/date pair, in the same order as the qualifying file) to demonstrate the full pipeline end to end, but it can't be scored against anything -- there's no answer key left to check it.

In [2]:
def count_movie_blocks_and_pairs(filepath):
    """Count movie headers vs. (movie, user) pairs in one of the two evaluation files."""
    movie_headers, pairs = 0, 0
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip().endswith(':'):
                movie_headers += 1
            else:
                pairs += 1
    return movie_headers, pairs

for fname in ['probe.txt', 'qualifying.txt']:
    headers, pairs = count_movie_blocks_and_pairs(os.path.join(DATA_PATH, fname))
    print(f'{fname:<15} {headers:>7,} movies   {pairs:>11,} (movie, user) pairs')

probe.txt        16,938 movies     1,408,395 (movie, user) pairs
qualifying.txt   17,470 movies     2,817,131 (movie, user) pairs


Every non-header line under a probe or qualifying movie id is just a customer id (plus a date for qualifying) -- no rating, because for probe the rating already lives inside the training files and gets predicted and compared against, and for qualifying it was never published at all. Probe covers 16,938 of the 17,770 total movies -- not quite all of them, meaning a small number of movies in the full catalog have no held-out row to evaluate against and will only ever be judged by however well the model fits their training ratings.

## 3. Planned System: Temporal-Bias-Adjusted ALS, Blended with a Non-Spark Baseline

The core method builds directly on Project 5's ALS-on-Spark approach, with one addition: a time-aware bias correction fit before the factorization step, plus a second model blended in afterward.

**Why temporal matters here specifically:** ratings aren't stationary over six-plus years of data. Average ratings drift, individual users' tastes shift, and Netflix's own rating interface changed at least once during the collection window in ways that show up as a real jump in the data rather than organic taste change. A model that treats every rating as if it came from the same stable process is throwing away a real, measurable signal.

**Step 1 -- Temporal bias correction.** Before fitting any factorization model, compute time-aware bias terms: a global rating trend over time, a per-movie bias that's allowed to drift across coarse time windows (movies accumulate enough ratings for binned drift to mean something), and a per-user bias that drifts as a continuous function of days since that user's own mean rating date (individual users rate too sparsely for per-user time bins to be meaningful, so a smooth per-user trend is the more honest choice). Subtracting this bias from every rating before factorization, then adding it back at prediction time, is the plan -- not full per-user latent-factor drift, which is a much heavier lift for the time available.

**Step 2 -- ALS on the bias-adjusted residual.** Same Spark ALS approach proven out in Project 5, fit on what's left over after the temporal bias is removed, with rank and regularization re-tuned specifically for the full dataset's scale rather than reusing Project 5's smaller-subset values directly -- the regParam lesson from Project 5 (a value tuned for one scale/optimizer doesn't automatically transfer) applies at least as much here, given how much larger and sparser the full corpus is than any subset used so far.

**Step 3 -- Blend with a non-Spark baseline.** A `surprise`-based SVD model, in the same family as Project 3's baseline, gets fit independently and blended with the temporal-ALS predictions using a regression fit on a held-out validation slice (not probe itself) -- learning how much to trust each model rather than averaging them blindly.

**Explicitly out of scope:** an implicit-feedback signal (which movies a user rated at all, regardless of the rating given) and full per-user latent-factor drift are both real techniques for pushing RMSE further down, but both add substantial implementation risk for the timeline available and aren't part of this plan. Noted here as a direction for future work, not attempted.

## 4. Experimental Design: How This Gets Tested, Not Just Assumed

Every claim about what helps needs to be measured on this project's own data, not assumed from how the technique is supposed to work in general. The plan:

**Data splits.** Training pool = all four `combined_data` files, minus every pair that appears in `probe.txt`. A further validation slice gets carved out of that training pool (not from probe) for tuning rank/regularization and fitting the blend weights. `probe.txt` itself stays untouched until the very last step, so the final reported RMSE is genuinely held-out -- nothing used to make a tuning decision ever touches the number that gets reported.

**Ablation ladder.** Rather than jumping straight to the full pipeline, each component gets measured on top of the last one, so the writeup can show exactly what each piece contributed instead of just asserting it:
1. Global mean predictor (trivial baseline, sanity check)
2. Static user + item bias (the Project 1 approach, now at full scale)
3. Temporal bias (same as #2, but with the time-drift terms added) -- isolates what temporal modeling alone contributes, before any factorization
4. Plain ALS on the full data, no temporal adjustment, rank/regParam swept on the validation slice
5. Temporal-bias-adjusted ALS -- isolates what the factorization step adds on top of temporal bias alone
6. Final blend with the non-Spark SVD baseline

**Rank/regularization sweep.** A small grid on the validation slice, not reused values from Project 5 -- the full dataset's scale and sparsity profile are different enough that the previously-tuned configuration isn't a safe assumption.

**Secondary metrics.** Precision@k, recall@k, and catalog coverage carried forward from Projects 4 and 5, computed on the same held-out probe predictions, as a check on whether RMSE improvements actually translate into better-looking recommendations.

## 5. Platform and Compute Plan

The factorization step runs on Spark, continuing directly from Project 5's approach, but at a scale where `local[*]` mode is a real risk rather than a safe assumption -- the full corpus is roughly five times the size of Project 5's largest subset. The plan is to first confirm the full pipeline's correctness at a smaller, fast-iterating scale (the same staged-sizing approach used across this whole project series), then run the full 100-million-row fit on **Azure Databricks** rather than a single laptop -- this project's own training and modeling platform, chosen specifically to remove the scale risk local Spark would otherwise carry at this size.

That is a separate decision from Project 6's requirement, and the two aren't the same thing: **the model itself is trained and run on Azure Databricks; AWS is used only for Project 6's specific deployment piece**, built around the already-trained model rather than as the platform it runs on. Project 6's own instructions explicitly allow submitting both together, so this project folds that AWS deployment in as a second, separate artifact -- S3 for long-term storage of the trained model, a Lambda function for serving predictions on request, and a VPC restricting access to just this project. One five-minute video is planned to cover both the recommender's results and the AWS deployment walkthrough, submitted to both the Final Project and Project 6 dropboxes.

## 6. What Counts as a Good Result Here

The Netflix Prize's own winning RMSE of 0.8567 has been the target line on every RMSE chart across this entire project series, and it stays that way here -- but it's worth being direct about what that number actually represents: the final blended result of a team effort combining well over one hundred separate models, refined over roughly three years. It is not a realistic bar for a single method built by one person in a few weeks, and treating it as a pass/fail line would set up a misleading comparison.

A more honest framing: plain matrix factorization on the full, unfiltered corpus is expected to start out worse than Project 5's curated-subset result of 0.8643, not better, simply because the full corpus reintroduces a huge amount of long-tail sparsity (many movies and users with very few ratings each) that Project 5's popularity-filtered subset avoided by construction. The temporal bias correction and the blend with the SVD baseline are the two levers expected to claw back a meaningful part of that gap. Landing meaningfully closer to 0.8567 than plain, untuned matrix factorization would -- while being transparent about the distance still remaining -- is the actual goal, not crossing the line outright.

## 7. Known Limitations Going In

**Scale risk.** The full dataset is roughly five times larger than anything run so far in this series. Local Spark handled Project 5's scale comfortably; whether it handles this scale without a cloud platform is a real open question, not a settled one.

**Temporal bias granularity is a judgment call.** Coarser movie-drift bins are more stable but miss faster-moving trends; finer bins risk overfitting sparse windows. The plan uses coarse-ish binning for movies and a smooth per-user trend rather than per-user bins, but the actual bin width is a tuning decision to make empirically, not something settled in advance.

**Blend weights need a genuinely separate validation slice.** Fitting the blend on the same data used to tune ALS's rank and regularization risks a falsely optimistic result; the plan keeps a dedicated validation slice for exactly this reason, separate from both the ALS tuning step and the final probe evaluation.

**No implicit-feedback signal.** Whether a user rated a movie at all (regardless of the rating given) carries real signal that this plan doesn't capture. Left as a direction for future work rather than attempted here, given the added implementation complexity relative to the time available.

**`coldStartStrategy='drop'` is more likely to matter at this scale than it did in Project 5.** Project 5's curated subset saw zero rows dropped, because every user had at least 20 ratings and every movie had thousands. The full corpus includes users and movies with only one or two ratings each, so a nonzero drop rate is plausible here and needs to be measured and reported honestly rather than assumed away.

## 8. Timeline

- **Now - Jul 8:** Build and validate the parsing/split pipeline at reduced scale; confirm the ablation ladder runs end to end before committing to the full 100-million-row fit.
- **Jul 8-12:** Run the full-scale ablation ladder and rank/regularization sweep; finalize this planning document (soft target: before the Jul 8 Zoom meetup, per the professor's request; official deadline Jul 12, 1:00 PM ET).
- **Jul 12-18:** Full implementation notebook trained and run on Azure Databricks; Project 6's AWS deployment (S3 + Lambda + VPC) built around that trained model, not as its training platform; combined presentation/deployment video, due Jul 18, 11:59 PM ET.